# Лекция 3. Дерево поиска: искать любой ключ, а не только минимум

> Конспект третьего занятия курса «Алгоритмы и структуры данных» (ДПО).
> Занятие начинается с возврата к куче и переходит к дереву поиска.
> Куча отдаёт минимум, но больше ничего не умеет, и это задаёт вопрос всей лекции:
> какой порядок нужно потребовать от дерева, чтобы быстро находить любой ключ.
> Ответ это дерево поиска и его операции: поиск, вставка, удаление и обход по порядку,
> который даёт сортировку деревом. Поиск, вставка и удаление держатся на глубине дерева.
> Поэтому в конце лекция разбирает, чем эту глубину чинят: поворотом и сбалансированными деревьями.

**После лекции вы сможете:**

- объяснить, какое требование к порядку отличает дерево поиска от кучи;
- написать вершину со ссылками, поиск, вставку, обход по порядку и удаление;
- отсортировать числа деревом и сказать, чем это отличается от пирамидальной сортировки;
- объяснить, почему поиск, вставка и удаление стоят столько, сколько вершин на пути от корня, и как отсортированный ввод вытягивает дерево в цепочку;
- сказать, что делает поворот, и чем AVL, красно-чёрное и splay-дерево отличаются друг от друга.

**Что нужно знать:** лекция 2: дерево, корень, предок, потомок, лист, поддерево, глубина, оценки `O(log n)` и `O(n · log n)`, модуль `heapq`, пирамидальная сортировка. Лекция 1: бинарный поиск, `log2`, индекс и B-дерево из раздела 3. Python: функции, списки, рекурсия. Класс из одного метода `__init__` объясняется в разделе 3, заранее знать его не нужно.

**Время:** 35 минут на чтение. Около двух часов, если запускать код и делать все предсказания. Если читаете за два раза, удобная пауза после раздела 6. Вернувшись, перезапустите ячейки разделов 3, 4, 5 и 6. В них живут `Node`, `show`, `depth`, `find`, `path`, `add`, `build`, `DEMO`, `find_min` и `in_order`, дальше лекция пользуется ими постоянно.

**Как читать.** Ячейки запускайте по порядку, сверху вниз. Блоки «Предскажите» и «Попробуйте сами» это барьеры: сначала отвечаете сами, потом открываете спойлер. Спойлеры обведены горизонтальными линиями, внутри них лежат ответы и анимации. Анимации это файлы из папки `img` рядом с ноутбуком. Если открыли ноутбук в Colab, картинок не будет, смотрите их на GitHub. Ввода с клавиатуры в этой лекции нет, все ячейки выполняются сами.

---

## 1. Чего не умеет куча

На прошлом занятии куча решила свою задачу. Она отдаёт минимум за `O(log n)` и принимает новый элемент за столько же. Третье занятие начинается с возврата к ней, и первый вопрос такой: а что куча умеет, кроме минимума?

Соотношение порядка в куче требует одного. Каждый предок меньше своих потомков. Про двух потомков одной вершины оно не говорит ничего, и про соседние поддеревья тоже ничего.

**Предскажите.** Вот куча на минимум из шести чисел: `[12, 25, 30, 40, 70, 55]`. Вы ищете в ней число `50`. С корня видно, что `50` больше `12`. В какое поддерево спускаться, левое или правое?

---

<details><summary>Ответ</summary>

Ни в какое. Спускаться некуда, потому что подсказки нет. Слева от корня стоит `25`, справа `30`, и `50` может оказаться под любым из них. В этой куче его нет вовсе, но узнать это можно только просмотрев все шесть чисел.

Куча складывает элементы так, что маленькие оказываются выше. Этого хватает, чтобы забрать минимум. Для поиска произвольного числа этого мало. Под вершиной, которая меньше искомого, надо проверять оба поддерева. В худшем случае проверка обходит всю кучу.
</details>

---

Запустите и сверьтесь. Куча собрана модулем `heapq` из лекции 2.

In [ ]:
from heapq import heapify

keys = [40, 12, 55, 25, 70, 30]
heapify(keys)
print("куча:", keys)
print("минимум за один взгляд:", keys[0])

# ищем 50 и 55: сравнивать приходится со всеми числами подряд
for key in (50, 55):
    seen = 0
    found = False
    for value in keys:
        seen += 1
        if value == key:
            found = True
            break
    print(f"{key}: найдено {found}, просмотрено {seen} из {len(keys)}")

## Второе, чего куча не умеет: выдать элементы по возрастанию. В лекции 2 это делала пирамидальная сортировка. Она забирала минимум `n` раз и тем самым разбирала кучу до конца.

Итого куча отвечает на один вопрос: «кто сейчас минимальный». Вопросы «есть ли здесь ключ `57`», «какой ключ идёт следующим после `57`», «перечисли всё по возрастанию» ей недоступны. Значит, от дерева надо потребовать порядок другой формы.

---

## 2. Дерево поиска: одно требование к порядку

Слова из лекции 2 действуют дальше без изменений: дерево, корень, предок, потомок, лист, глубина. Понадобится и **поддерево** (subtree) из раздела 11 лекции 2. Это вершина вместе со всем, что висит под ней.

Новое требование такое. Для каждой вершины все ключи её левого поддерева меньше её ключа, а все ключи правого поддерева больше. Дерево с таким требованием называют **деревом поиска** (binary search tree, BST).

Разница с кучей в двух словах. Куча упорядочивает вершину против её потомков, сверху вниз. Дерево поиска упорядочивает поддеревья слева направо.

Вот дерево поиска из семи ключей:

```
                50
             /      \
          30          70
         /  \        /  \
       20    40    60    80
```

![Дерево поиска из семи ключей](img/bst_tree.png)

Проверьте требование на вершине `30`. Слева от неё `20`, справа `40`, и `20 < 30 < 40`. Теперь на корне. В левом поддереве корня лежат `20`, `30`, `40`, и все они меньше `50`. В правом лежат `60`, `70`, `80`, и все больше. Требование выполняется в каждой вершине, а не только у корня. Именно оно даёт право на каждом шаге откидывать целое поддерево.

**Предскажите.** Вы добавляете в это дерево ключ `45`. Место для него только одно. Найдите его и скажите, какой глубины станет дерево.

---

<details><summary>Ответ</summary>

`45` меньше `50`, значит, идёт налево. Больше `30`, значит, направо. Больше `40`, значит, снова направо. Место справа от `40` свободно, там `45` и повиснет. Дерево станет глубиной 4, а путь до новой вершины такой: `50`, `30`, `40`, `45`.
</details>

---

Чтобы проверить такие рассуждения кодом, дерево надо как-то хранить в памяти.

---

## 3. Вершина и ссылки: дерево в Python

Кучу в лекции 2 удалось положить в список, потому что она почти полное дерево. Уровни у неё заполнены слева направо без пропусков, поэтому предок находится арифметикой: `(i - 1) // 2`.

У дерева поиска такого права нет. Его форму определяет порядок, в котором пришли ключи, и пропуски в уровнях обычное дело. Посмотрите на рисунок выше и добавьте `45`: уровень 4 занят одной вершиной из восьми возможных. Список для такого дерева пришлось бы держать почти пустым.

Поэтому вершина хранит ссылки на своих потомков прямо в себе. Нужен объект с тремя полями: ключ, левый потомок, правый потомок. В Python такой объект описывают классом.

In [ ]:
class Node:
    """Вершина дерева поиска: ключ и две ссылки."""

    def __init__(self, key):
        self.key = key
        self.left = None       # левое поддерево: ключи меньше
        self.right = None      # правое поддерево: ключи больше


# соберём дерево с рисунка вручную, чтобы увидеть сами ссылки
root = Node(50)
root.left = Node(30)
root.right = Node(70)
root.left.left = Node(20)
root.left.right = Node(40)
root.right.left = Node(60)
root.right.right = Node(80)

print("корень:", root.key)
print("его потомки:", root.left.key, "и", root.right.key)
print("левый потомок левого потомка:", root.left.left.key)
print("у листа потомков нет:", root.left.left.left, root.left.left.right)

Как это читать. Запись `Node(50)` создаёт новый объект и сразу вызывает его метод `__init__` с `key = 50`. Внутри метода `self` это сам создаваемый объект. Строка `self.key = key` кладёт ключ в его поле `key`. Думайте об объекте как о словаре с тремя постоянными полями: `root.key` работает как `root["key"]`.

Цепочка точек читается слева направо. `root.left` это левый потомок корня. `root.left.left` это левый потомок этого потомка, то есть вершина `20`.

`None` в поле означает «потомка нет». У листа оба поля равны `None`, и это же значение служит признаком пустого места при вставке.

Дерево удобно видеть целиком, поэтому пригодится печать. Рисовать ветки в тексте сложно, поэтому дерево печатается лежащим на боку. Корень стоит у левого края, каждый уровень сдвинут вправо, правое поддерево оказывается сверху, левое снизу. Наклоните голову влево и получите привычную картинку.

In [ ]:
def show(node, level=0):
    """Печатает дерево на боку: корень слева, правое поддерево сверху."""
    if node is None:
        return
    show(node.right, level + 1)
    print("     " * level + str(node.key))
    show(node.left, level + 1)


def depth(node):
    """Глубина: число уровней в самой длинной ветке."""
    if node is None:
        return 0
    return 1 + max(depth(node.left), depth(node.right))


show(root)
print("глубина:", depth(root))

Обе функции рекурсивные, и обе устроены одинаково. Функция вызывает себя для левого и правого поддерева, а останавливается на `None`. Это та же остановка, что у `sift_down` в лекции 2: рекурсия заканчивается, когда идти дальше некуда.

---

## 4. Поиск: спуск от корня

Дерево собрано и видно целиком. Теперь можно проверить обещание раздела 2: одно сравнение отбрасывает целое поддерево. Поиск ключа не смотрит на оба поддерева, он смотрит на одно.

Правило спуска такое. Сравните искомый ключ с ключом вершины. Равны, значит, нашли. Меньше, значит, идите налево, потому что справа все ключи больше. Больше, значит, направо. Кончились вершины, значит, ключа в дереве нет.

In [ ]:
def find(root, key):
    """Есть ли ключ в дереве."""
    node = root
    while node is not None:
        if key == node.key:
            return True
        node = node.left if key < node.key else node.right
    return False


def path(root, key):
    """Ключи вершин, через которые прошёл поиск. Нужен, чтобы видеть путь."""
    node, seen = root, []
    while node is not None:
        seen.append(node.key)
        if key == node.key:
            return seen
        node = node.left if key < node.key else node.right
    return seen


for key in (60, 20, 45):
    print(f"{key}: найдено {find(root, key)}, путь {path(root, key)}")

Разберите путь к `60` по шагам. Дерево то же, что на рисунке.

| Шаг | Вершина | Сравнение | Куда дальше | Что отброшено |
|---|---|---|---|---|
| 1 | `50` | `60 > 50` | направо | всё левое поддерево: `20`, `30`, `40` |
| 2 | `70` | `60 < 70` | налево | `80` |
| 3 | `60` | `60 == 60` | нашли | — |

Три вершины на дерево из семи ключей. Отброшенные вершины никто не смотрел: они отпали целыми поддеревьями.

Поиск `45` устроен так же, но кончается иначе. Путь `50`, `30`, `40`. На `40` надо идти направо, а там пусто. Значит, `45` в дереве нет, и на это ушли те же три вершины.

Дальше цену поиска считаем в вершинах. На каждой вершине код делает до двух сравнений, `==` и `<`, но растёт цена вместе с числом вершин на пути.

**Предскажите.** Сколько вершин посмотрит поиск в дереве глубины 4, если ключа в дереве нет? А если в дереве миллион ключей и заполнены все уровни, кроме, может быть, последнего?

---

<details><summary>Ответ</summary>

Не больше четырёх. Каждая вершина спускает поиск на один уровень, а уровней четыре. Путь может закончиться и раньше, если упрётся в `None` выше последнего уровня.

Для миллиона ключей глубина равна 20, потому что `2^20` чуть больше миллиона. Значит, не больше 20 вершин. Это тот же `log2(n)`, что в лекции 1 у бинарного поиска и в лекции 2 у кучи. Цена поиска равна длине пути от корня, а длина пути не больше глубины: `O(log n)` для дерева с заполненными уровнями.
</details>

---

Анимация показывает тот же спуск к `60`. Синим отмечена вершина, на которую смотрит поиск, серым отброшенные поддеревья, зелёным найденная вершина.

---

<details><summary>Анимация: поиск ключа 60</summary>

![Поиск ключа в дереве поиска](img/search.gif)

</details>

---

## 5. Вставка: туда же, куда искали

Место новой вершины подсказал раздел 2. Ключ идёт туда, где закончился бы его неудачный поиск.

Значит, вставка это тот же спуск. Отличается она концом. Поиск, упёршись в `None`, возвращает «нет», а вставка ставит на это место новую вершину.

In [ ]:
def add(root, key):
    """Вставить ключ. Возвращает корень дерева."""
    if root is None:
        return Node(key)
    node = root
    while True:
        if key < node.key:
            if node.left is None:
                node.left = Node(key)
                return root
            node = node.left
        elif key > node.key:
            if node.right is None:
                node.right = Node(key)
                return root
            node = node.right
        else:
            return root            # такой ключ уже есть, второй раз не храним


def build(keys):
    """Дерево из последовательности ключей, по одному."""
    root = None
    for key in keys:
        root = add(root, key)
    return root


DEMO = [50, 30, 70, 20, 40, 60, 80]
root = build(DEMO)
print("дерево из", DEMO)
show(root)
print("глубина:", depth(root))

root = add(root, 45)
print()
# в новом дереве 45 уже есть, поэтому путь смотрим по дереву без него
print("после add(45): путь поиска был", path(build(DEMO), 45))
show(root)
print("глубина:", depth(root))

Обратите внимание на порядок ключей в `DEMO`. Корнем стал `50`, потому что он пришёл первым. Форма дерева зависит от порядка ввода целиком, и в разделе 9 это выйдет боком.

Ещё одна деталь в `add`. Ветка `else` не делает ничего. Дубликаты дерево не хранит, потому что требование к порядку говорит про «меньше» и «больше», а равным ключам места не оставляет. В куче дубликаты лежали спокойно, здесь их пришлось решать отдельно. Другие реализации вешают счётчик на вершину или отправляют равные ключи всегда направо.

---

<details><summary>Анимация: вставка ключа 45</summary>

![Вставка ключа в дерево поиска](img/insert.gif)

</details>

---

Вставкой дерево наполняется. Теперь спросите его о том, чего куча не умела: выдать все ключи по порядку и остаться целым.

---

## 6. Минимум, максимум и сортировка деревом

Сначала крайние ключи. Куча отдавала минимум мгновенно, потому что он лежал в корне. В дереве поиска корень не минимум. Зато минимум находится без единого сравнения ключей.

Идите от корня всё время налево. Слева лежат ключи поменьше, поэтому последняя вершина перед `None` и есть минимальный ключ. Максимум симметрично: всё время направо. Бесплатным минимум от этого не становится. Путь до крайней левой вершины тоже спуск, и он не длиннее глубины.

In [ ]:
def find_min(root):
    while root.left is not None:
        root = root.left
    return root.key


def find_max(root):
    while root.right is not None:
        root = root.right
    return root.key


print("минимум:", find_min(root), "| максимум:", find_max(root))

Теперь главное умение, которого у кучи не было. Дерево поиска умеет выдать все ключи по возрастанию, ничего при этом не разрушая.

Смотрите на требование к порядку ещё раз. В левом поддереве всё меньше вершины, в правом всё больше. Значит, порядок такой: сначала целиком левое поддерево, потом сама вершина, потом целиком правое. Тот же рецепт применяется внутри каждого поддерева, то есть это рекурсия. Такой обход называют **обходом по порядку** (in-order traversal).

In [ ]:
def in_order(node, keys):
    """Ключи по возрастанию. Собирает их в список keys."""
    if node is None:
        return keys
    in_order(node.left, keys)      # сначала всё, что меньше
    keys.append(node.key)          # потом сама вершина
    in_order(node.right, keys)     # потом всё, что больше
    return keys


print("обход по порядку:", in_order(root, []))

Список `keys` один на все вызовы. Каждый вызов дописывает в него ключи своего поддерева. Поэтому снаружи передают пустой список `[]` и получают его же, уже заполненный.

**Предскажите.** Возьмите двенадцать случайных чисел, постройте из них дерево и сделайте обход по порядку. Что получится, и сколько это стоит по сравнению с пирамидальной сортировкой из лекции 2?

---

<details><summary>Ответ</summary>

Получится отсортированный список. Это **сортировка деревом** (tree sort), обещанная в конце лекции 2.

Цена складывается из двух частей. Построение это `n` вставок, каждая не дороже глубины. Обход заходит в каждую вершину один раз, это `O(n)`. Если дерево не вытянулось в цепочку, глубина порядка `log n`, и вместе выходит `O(n · log n)`, как у пирамидальной сортировки. Если вытянулось, вставки становятся дорогими, и сортировка стоит `O(n²)`. Когда так бывает, покажет раздел 9.

Главная разница в том, что остаётся после. Пирамидальная сортировка разбирает кучу и отдаёт список. Дерево остаётся стоять, и его можно спрашивать дальше: искать ключ, добавлять новый, снова выдавать всё по порядку. Именно за это платят ссылками и памятью на вершины.
</details>

---

In [ ]:
import random

random.seed(7)
numbers = random.sample(range(10, 99), 12)
tree = build(numbers)
print("вход: ", numbers)
print("обход:", in_order(tree, []))
print("совпало с sorted:", in_order(tree, []) == sorted(numbers))

Дерево умеет искать, вставлять и выдавать всё по порядку. Осталась последняя операция: убрать ключ.

---

## 7. Удаление: три случая

Поиск и вставка трогали одну вершину. Удаление сложнее, потому что убранная вершина оставляет за собой дыру, и поддеревья из этой дыры надо куда-то деть.

Разберите три случая на дереве из раздела 5.

1. **У вершины нет потомков.** Лист просто исчезает. Так уходит `20`.
2. **У вершины один потомок.** На её место встаёт этот потомок вместе со всем своим поддеревом. Требование к порядку не нарушится, потому что поддерево и так лежало с правильной стороны. Так уходит `40`: его единственный потомок `45`.
3. **У вершины два потомка.** Убрать её нельзя, дыру нечем закрыть, поэтому вершину не убирают. В неё переписывают другой ключ, а убирают уже его старое место. Подходят два ключа: максимум левого поддерева или минимум правого. Это соседи удаляемого ключа по порядку, поэтому любой из них встанет на его место законно.

Третий случай сводится к первым двум. Минимум правого поддерева не имеет левого потомка по определению. Значит, его удаление это случай 1 или 2.

In [ ]:
def remove(node, key):
    """Удалить ключ. Возвращает корень поддерева после удаления."""
    if node is None:
        return None
    if key < node.key:
        node.left = remove(node.left, key)
    elif key > node.key:
        node.right = remove(node.right, key)
    else:
        if node.left is None:              # нет левого: лист (вернётся None) или случай 2
            return node.right
        if node.right is None:             # нет правого: случай 2
            return node.left
        node.key = find_min(node.right)    # случай 3: минимум правого поддерева
        node.right = remove(node.right, node.key)
    return node


for key, case in ((20, "лист"), (40, "один потомок"), (30, "два потомка")):
    tree = build(DEMO + [45])
    tree = remove(tree, key)
    print(f"убрали {key} ({case}): {in_order(tree, [])}")

Главная строка тут `node.left = remove(node.left, key)`. Вызов `remove` для поддерева возвращает то, что должно висеть на этом месте после удаления. Если удаление внутри поддерева ничего не поменяло сверху, вернётся та же вершина, и присваивание ничего не сломает. Если удалили саму вершину, вернётся её замена или `None`, и присваивание перевесит ссылку.

Посмотрите на удаление `30` подробно. Это третий случай.

In [ ]:
tree = build(DEMO + [45])
print("до удаления 30:")
show(tree)
tree = remove(tree, 30)
print()
print("после удаления 30: на его месте 40, минимум правого поддерева")
show(tree)
print("обход по порядку:", in_order(tree, []))

Вот все вызовы `remove` для этого удаления, сверху вниз.

| Вызов | Что видит | Что делает | Что возвращает |
|---|---|---|---|
| `remove(50, 30)` | `30 < 50` | идёт налево: `50.left = remove(30, 30)` | вершину `50` |
| `remove(30, 30)` | ключ найден, два потомка | пишет в вершину `40`, минимум справа, затем `right = remove(40, 40)` | ту же вершину, теперь с ключом `40` |
| `remove(40, 40)` | ключ найден, левого потомка нет | случай 2 | вершину `45` |

Поднимаясь обратно, каждый вызов вешает возвращённое на своё место. Старая вершина `40` пропала, потому что на её место в правом поле встала `45`.

Поэтому `remove` и возвращает корень, и принимает его же. Удаление может сменить сам корень. Это случается, когда у корня не больше одного потомка. Тогда на его место встаёт этот потомок, а если потомков нет, дерево становится пустым.

### Где подведёт: забыть присвоить результат

In [ ]:
tree = build(DEMO)
remove(tree, 20)                       # результат никуда не записали
print("после remove без присваивания:", in_order(tree, []))

tree = build([50, 70])
remove(tree, 50)                       # у корня один потомок: корень должен смениться
print("убрали корень 50 без присваивания:", in_order(tree, []))

tree = build([50, 70])
tree = remove(tree, 50)
print("то же самое с присваиванием:", in_order(tree, []), "| корень теперь", tree.key)

Первый вызов сработал, хотя результат выбросили. Дело в том, что `20` был листом внутри дерева, и `remove` изменила поле `left` у вершины `30` по ссылке. Второй вызов не сработал. `50` был корнем с одним потомком, функция вернула вершину `70` как новый корень. Но переменная `tree` осталась смотреть на старую вершину `50`. Ошибка такого рода тем и неприятна, что проявляется не всегда. Пишите `tree = remove(tree, key)` всегда.

---

## 8. Зачем это учить: запросы по порядку

У дерева поиска теперь есть все операции. Посмотрите, где они работают за пределами лекции. Вот таблица сотрудников в базе данных и три запроса к ней.

```sql
SELECT * FROM employees WHERE id = 8342;                -- один ключ
SELECT * FROM employees WHERE salary BETWEEN 90 AND 150; -- диапазон
SELECT * FROM employees ORDER BY salary LIMIT 20;        -- первые по порядку
```

Сравните три структуры. Первая это **хеш-таблица** (hash table), она стоит внутри `dict` в Python. Хеш-таблица пропускает ключ через хеш-функцию того же рода, что в лекции 1. Только эта функция быстрая и без защиты от подбора. Хеш даёт номер ячейки, и ключ ищут сразу в ней, в среднем за один шаг. Вторая это куча из лекции 2, третья дерево поиска.

**Предскажите.** На какие из трёх запросов каждая структура ответит, не просматривая всю таблицу?

---

<details><summary>Ответ</summary>

| Структура | Один ключ | Диапазон | Первые 20 по порядку |
|---|---|---|---|
| Хеш-таблица | да | нет | нет |
| Куча | нет | нет | да |
| Дерево поиска | да | да | да |

Хеш ломает порядок нарочно: соседние зарплаты лежат в случайных ячейках. Поэтому диапазон и «первые по порядку» хеш-таблице недоступны, ей пришлось бы читать всё.

Куча отвечает только на третий запрос. Двадцать извлечений стоят `O(20 · log n)`, в `heapq` это делает `nsmallest`. Найти один `id` или диапазон она без полного просмотра не может.

Дерево поиска отвечает на все три. Первый запрос это спуск из раздела 4. Третий это обход по порядку, остановленный на двадцатой вершине. Второй тоже обход, но заходить надо не во все поддеревья. Если ключ вершины меньше нижней границы, её левое поддерево целиком мимо, и его можно не открывать.
</details>

---

В разделе 3 лекции 1 вы видели B-дерево (B-tree): страницы по 500 ключей и оглавления над ними. Теперь видно, почему оно называется деревом. Это дерево поиска, у которого в каждой вершине сотни ключей по порядку. Порядок даёт ответы на все три запроса. Сотни ключей в вершине нужны, чтобы одно чтение с диска приносило целую страницу.

Во всех трёх ответах дерево спускается от корня, и цена равна длине пути. Следующий раздел проверяет, какой длины бывает этот путь.

---

## 9. Где подведёт: отсортированный ввод

Поиск, вставка и удаление стоят столько, сколько вершин на пути от корня. Путь не длиннее глубины. Пора проверить, какой бывает глубина.

**Предскажите.** Вы строите дерево из чисел `0, 1, 2, ..., 1999`, подавая их по возрастанию. Какой формы получится дерево и сколько вершин пройдёт поиск числа `1999`?

---

<details><summary>Ответ</summary>

Каждое следующее число больше всех предыдущих, поэтому каждое уходит направо. Левых потомков не появляется ни у кого. Дерево вытягивается в **цепочку** (degenerate tree) из 2000 вершин, и поиск `1999` проходит все 2000.

Требование к порядку при этом не нарушено ни в одной вершине. Дерево законное, просто бесполезное: `O(log n)` превратилось в `O(n)`.
</details>

---

In [ ]:
line = build(range(2000))                        # по возрастанию
random.seed(3)
mixed = build(random.sample(range(2000), 2000))  # те же числа вперемешку

print("цепочка: путь к 1999 длиной", len(path(line, 1999)), "вершин")
print("вперемешку: путь к 1999 длиной", len(path(mixed, 1999)), "вершин, глубина дерева", depth(mixed))

У перемешанного дерева путь к `1999` короче его глубины. Глубина это самая длинная ветка, а конкретный ключ может лежать выше. Поэтому глубина это верхняя граница пути, а не точная цена.

Во времени та же картина.

In [ ]:
import time

per_search = {}
for name, tree in (("цепочка", line), ("вперемешку", mixed)):
    start = time.perf_counter()
    for _ in range(500):
        find(tree, 1999)
    per_search[name] = (time.perf_counter() - start) / 500 * 1_000_000
    print(f"{name}: один поиск за {per_search[name]:.1f} мкс")
print(f"цепочка медленнее в {per_search['цепочка'] / per_search['вперемешку']:.0f} раз")

У цепочки есть второе следствие, и оно бьёт по коду этой лекции. Обход по порядку рекурсивный, а рекурсия в Python ограничена примерно тысячей вложенных вызовов. На цепочке из 2000 вершин обход до дна не доходит.

In [ ]:
try:
    in_order(line, [])
    print("обход цепочки: прошёл")
except RecursionError:
    print("обход цепочки: RecursionError, рекурсия глубже лимита Python")

print("обход перемешанного дерева отсортирован:", in_order(mixed, []) == sorted(range(2000)))

Виноват не Python. Лимит только показывает проблему: цепочка из 2000 вершин ведёт себя как список, только с лишними полями.

Случайный порядок спасает, но не идеально. Посмотрите на глубину.

In [ ]:
import math

for n in (1000, 100_000):
    random.seed(1)
    tree = build(random.sample(range(n), n))
    ideal = math.ceil(math.log2(n + 1))
    print(f"n = {n}: глубина {depth(tree)}, у дерева с заполненными уровнями было бы {ideal}")

Случайное дерево в два с лишним раза глубже идеального. Порядок величины тот же, `O(log n)`, и это приемлемо. Но на порядок ввода полагаться нельзя. Данные приходят отсортированными сплошь и рядом: ключи по дате, автоинкрементный `id` из базы данных. Дерево должно держать глубину само.

---

## 10. Поворот: как чинят глубину

Главный инструмент починки называется **поворот** (rotation). Он меняет местами вершину с одним из её потомков.

Возьмите новое дерево, не из раздела 5: вершину `70` с левым потомком `50`. **Поворот вправо** (right rotation) делает `50` главной, а `70` её правым потомком. Правое поддерево `50` при этом переезжает: оно становится левым поддеревом `70`. Так и должно быть. Его ключи больше `50` и меньше `70`, а это в точности то место, которое освободилось.

In [ ]:
def rotate_right(node):
    """Поворот вправо вокруг node. Левый потомок становится главным."""
    pivot = node.left
    node.left = pivot.right    # то, что между ними по величине, переезжает
    pivot.right = node
    return pivot


chain = build([70, 50, 30])
print("до поворота, глубина", depth(chain), "обход", in_order(chain, []))
show(chain)

chain = rotate_right(chain)
print()
print("после поворота, глубина", depth(chain), "обход", in_order(chain, []))
show(chain)

Три строки сделали две вещи сразу. Глубина упала с 3 до 2, а обход по порядку не изменился. Второе важнее первого. Поворот перекладывает вершины, но не трогает порядок, поэтому дерево остаётся деревом поиска.

В этой цепочке у `50` не было правого поддерева, и переезжать было нечему. Проверьте поворот на дереве побольше, где переезд есть.

In [ ]:
big = build([70, 50, 80, 30, 60, 20])
before = in_order(big, [])
print("до поворота, глубина", depth(big))
show(big)

big = rotate_right(big)
print()
print("после поворота, глубина", depth(big), "| обход тот же:", in_order(big, []) == before)
show(big)

Следите за `60`. До поворота это правый потомок `50`, после поворота левый потомок `70`. Больше ничего не переехало, а глубина упала с 4 до 3.

---

<details><summary>Анимация: поворот вправо на дереве big</summary>

![Поворот вправо в дереве поиска](img/rotate.gif)

</details>

---

Поворот вправо поднимает левую сторону. **Поворот влево** (left rotation) симметрично поднимает правую. Устроен он так же, только `left` и `right` меняются местами.

Одного поворота мало. Он чинит глубину в одном месте, а дерево надо держать неглубоким всё время, после каждой вставки и каждого удаления. Значит, нужны правила: какую вершину проверять, по какому признаку считать её плохой и сколько поворотов делать. Деревья с такими правилами называют **сбалансированными** (balanced). На занятии говорили о трёх видах.

---

## 11. Три сбалансированных дерева

Все три держат глубину `O(log n)` при любом порядке ввода. Отличаются они тем, что хранят в вершине, насколько строго следят за балансом и когда делают повороты.

**AVL-дерево.** Название составлено из инициалов авторов, Адельсона-Вельского и Ландиса (1962). Каждая вершина помнит глубину своего поддерева. В других источниках её называют высотой (height). Правило: глубины левого и правого поддерева отличаются не больше чем на единицу. После вставки или удаления дерево идёт от изменённой вершины к корню и проверяет правило. Где оно нарушено, делается один поворот. Два поворота нужны, когда перекос идёт зигзагом: например, левый потомок сам перевешивает вправо. Тогда сначала поворачивают потомка, потом саму вершину.

Баланс у AVL самый строгий из трёх: глубина не больше примерно `1,44 · log2(n)`. Поэтому поиск самый быстрый. Плата тоже понятна. Строгое правило нарушается чаще, значит, перестроек больше. AVL выбирают, когда читают часто, а пишут редко.

**Красно-чёрное дерево** (red-black tree). Каждая вершина помнит свой цвет, красный или чёрный. Правила такие. Корень чёрный. У красной вершины оба потомка чёрные, то есть красная никогда не висит под красной. От любой вершины до любого пустого места (`None`) под ней по пути одинаковое число чёрных вершин.

Отсюда граница глубины. Самый короткий путь вниз состоит из одних чёрных вершин. Самый длинный чередует чёрные с красными, потому что две красные подряд не стоят. Значит, самый длинный путь не больше чем вдвое длиннее самого короткого. Отсюда глубина не больше `2 · log2(n + 1)`. Баланс слабее, чем у AVL, зато и перестроек меньше: часть нарушений чинится перекраской, без поворотов.

Из-за этого компромисса красно-чёрное дерево стоит внутри стандартных библиотек. Это `TreeMap` в Java и `std::map` в C++ во всех основных реализациях. В стандартной библиотеке Python такого дерева нет. `dict` и `set` это хеш-таблицы: `dict` помнит порядок вставки, но не порядок по величине.

**Splay-дерево** (splay tree). Не хранит ничего дополнительного. Правило у него другое: после каждого обращения к вершине она поворотами поднимается в корень. Найденный ключ оказывается наверху, и следующий запрос того же ключа стоит один шаг.

Отдельная операция при этом может оказаться дорогой, вплоть до `O(n)`. Но любая серия из `m` операций стоит не больше `O(m · log n)`, то есть `O(log n)` на операцию. Такую оценку называют **амортизированной** (amortized). Это не среднее по случайным данным, а гарантия для любой серии, как бы ни были выбраны запросы. Splay-дерево выигрывает, когда обращения неравномерны. Небольшая часть ключей запрашивается намного чаще остальных, и эти ключи сами собираются у корня.

| Дерево | Что хранит в вершине | Цена поиска | Чем платит | Где уместно |
|---|---|---|---|---|
| Дерево поиска без правил | ничего | до `n` | ничем, и потому ломается | когда порядок ввода заведомо случайный |
| AVL | глубину поддерева | до `1,44 · log2(n)` | больше всего перестроек | много поиска, мало изменений |
| Красно-чёрное | цвет | до `2 · log2(n + 1)` | поиск чуть медленнее, чем в AVL | общий случай, стандартные библиотеки |
| Splay | ничего | `O(log n)` на операцию в серии, одна может стоить `n` | оценка только амортизированная | часть ключей спрашивают намного чаще |

**Попробуйте сами.** Вы храните справочник, в котором 10 000 ключей. Запросы приходят так: 90 процентов обращений приходится на 50 ключей, остальные 10 процентов разбросаны по всему справочнику. Изменений почти нет. Какое дерево из трёх возьмёте?

---

<details><summary>Ответ</summary>

Splay-дерево. Оно само поднимет эти 50 ключей к корню, и почти каждый запрос закончится в нескольких шагах от него. AVL и красно-чёрное так не умеют. Они держат баланс по форме, а частоту запросов не учитывают. Путь к популярному ключу у них такой же, как к любому другому.

Если бы запросы были равномерными, а изменений по-прежнему мало, ответ был бы другим: AVL, у него самая маленькая глубина.
</details>

---

## 12. Проверьте себя

Ответьте без кода, потом сверьтесь.

1. Дерево поиска и куча оба бинарные деревья. Сформулируйте разницу в требованиях к порядку одной фразой каждое.
2. В дереве из раздела 5 (ключи `50, 30, 70, 20, 40, 60, 80`) ищут `65`. Какие вершины посмотрит поиск и чем закончится?
3. Почему обход по порядку выдаёт ключи по возрастанию? Объясните через требование к порядку, а не через код.
4. Из дерева удаляют вершину с двумя потомками. Почему на её место годится минимум правого поддерева, и почему его собственное удаление проще исходного?
5. Дерево построено из 100 000 ключей, пришедших по возрастанию. Сколько вершин пройдёт поиск последнего ключа? Что изменится, если те же ключи подать вперемешку?

---

<details><summary>Ответы</summary>

1. В куче каждый предок меньше своих потомков, порядок идёт сверху вниз. В дереве поиска все ключи левого поддерева меньше вершины, а правого больше, порядок идёт слева направо.
2. Посмотрит `50`, потом `70`, потом `60`. От `60` надо идти направо, там `None`, значит, `65` в дереве нет. Три вершины.
3. Для любой вершины все меньшие ключи лежат в её левом поддереве, а все большие в правом. Значит, правильная очередь такая: сначала левое поддерево целиком, затем вершина, затем правое. Внутри каждого поддерева работает то же правило, поэтому обход рекурсивный.
4. Минимум правого поддерева это следующий ключ по порядку после удаляемого. Все ключи левого поддерева меньше его, а все остальные в правом больше, поэтому требование к порядку сохраняется. Его удаление проще, потому что у минимума нет левого потомка: это случай листа или случай одного потомка.
5. Дерево вытянется в цепочку, поиск последнего ключа пройдёт все 100 000 вершин. Вперемешку путь не длиннее глубины. В разделе 9 для 100 000 ключей глубина вышла 40, в два с лишним раза больше идеальных 17. Значит, не больше 40 вершин, и разница в пути в две с половиной тысячи раз.
</details>

---

## 13. Итог

- Куча знает только минимум. Дерево поиска требует порядка слева направо, и это даёт поиск любого ключа, обход по возрастанию и запросы диапазоном.
- Вершина хранит ключ и две ссылки. Поиск, вставка, минимум и удаление устроены одним спуском от корня и стоят столько, сколько вершин на пути.
- Обход по порядку даёт сортировку деревом. На неглубоком дереве это `O(n · log n)`, как пирамидальная сортировка, но дерево после неё остаётся и отвечает на новые запросы.
- Отсортированный ввод вытягивает дерево в цепочку, и `O(log n)` становится `O(n)`. Глубину чинит поворот, а сбалансированные деревья решают, когда его делать: AVL строго, красно-чёрное мягче, splay поднимает к корню частые ключи.

**Где встретится дальше.** Обход по порядку один из обходов в глубину. Рядом с ним стоят прямой и обратный обходы (pre-order, post-order), а отдельно обход в ширину. Идея «сравнил и отбросил целую часть» встречается уже второй раз: сначала бинарный поиск в лекции 1, теперь дерево поиска.

**Что повторить через два-три дня.** Постройте по памяти дерево из ключей `60, 20, 80, 10, 40, 70, 90, 30`. Нарисуйте его, сделайте обход по порядку и проверьте, что получился отсортированный список. Затем удалите `20` и объясните вслух, почему на его место встала именно та вершина.